In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 2 - WEEK 12 BAYESIAN OPTIMISATION
# Run from inside week12/
# ============================================================
#
# Week 11 moved inward from the x2=1 boundary and performed
# substantially worse.
#
# Week 10 and the current incumbent both performed strongly
# very close to x2=1.
#
# Therefore Week 12:
# - refit the GP including Week 11
# - centre the search on the ACTUAL incumbent
# - use a conservative local region
# - explicitly include the x2=1 boundary as a diagnostic
# - compare posterior mean, EI and UCB
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 12 data
# ------------------------------------------------------------

X = np.load(
    "function2/initial_inputs.npy"
)

Y = np.load(
    "function2/initial_outputs.npy"
).reshape(-1)

best_idx = np.argmax(Y)

best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(
    best_x,
    "->",
    best_y
)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 11 calibration check
# ------------------------------------------------------------
#
# Week 11 selected:
# [0.702314, 0.972050]
#
# Prior Week 11 prediction was approximately:
# mean = 0.626126
# std  = 0.057153
#
# Actual:
# 0.5895778573315174
# ------------------------------------------------------------

week11_pred_mean = 0.626125645
week11_pred_std = 0.057152756

week11_actual = (
    0.5895778573315174
)

week11_error = (
    week11_actual
    - week11_pred_mean
)

week11_z_error = (
    week11_error
    / week11_pred_std
)

print("\n================================")
print("WEEK 11 CALIBRATION CHECK")
print("================================")

print(
    "Predicted mean:",
    week11_pred_mean
)

print(
    "Predicted std :",
    week11_pred_std
)

print(
    "Actual        :",
    week11_actual
)

print(
    "\nPrediction error:",
    week11_error
)

print(
    "\nError / predicted std:",
    week11_z_error
)


# ------------------------------------------------------------
# 3. Recent boundary evidence
# ------------------------------------------------------------

print("\n================================")
print("RECENT HIGH-PERFORMING POINTS")
print("================================")

print(
    "Current incumbent:",
    np.array([
        0.711177,
        1.000000
    ]),
    "->",
    0.679662733033218
)

print(
    "\nWeek 10:",
    np.array([
        0.702316,
        0.995076
    ]),
    "->",
    0.6682000915555967
)

print(
    "\nWeek 11:",
    np.array([
        0.702314,
        0.972050
    ]),
    "->",
    0.5895778573315174
)


# ------------------------------------------------------------
# 4. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(
            1e-3,
            1e3
        )
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(
            0.01,
            2.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(
            1e-8,
            1e-1
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(
    gp.kernel_
)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0
    / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print(
    "\nARD lengthscales:"
)
print(
    lengthscales
)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(
    relative_sensitivity
)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu
        - best_y
        - xi
    )

    valid = (
        sigma > 1e-12
    )

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask]
    - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

# Keep Week 12 fairly local.
empirical_cap = min(
    1.5 * nearest_distance,
    0.04
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print(
    "Nearest observed point:",
    nearest_distance
)

print(
    "\nEmpirical cap:",
    empirical_cap
)


# ------------------------------------------------------------
# 7. ARD-informed trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.20 * lengthscales,
    0.012,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x
    - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x
    + trust_half_width
)

print("\n================================")
print("WEEK 12 TRUST REGION")
print("================================")

print("Centre:")
print(
    best_x
)

print(
    "\nHalf-widths:"
)
print(
    trust_half_width
)

print(
    "\nLower:"
)
print(
    lower
)

print(
    "\nUpper:"
)
print(
    upper
)


# ------------------------------------------------------------
# 8. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(
    42
)

# Dense local box around incumbent
local_candidates = rng.uniform(
    lower,
    upper,
    size=(
        300000,
        2
    )
)


# Explicit x2 = 1 boundary profile
x1_boundary_values = np.linspace(
    lower[0],
    upper[0],
    50000
)

boundary_candidates = np.column_stack([
    x1_boundary_values,
    np.ones_like(
        x1_boundary_values
    )
])


# Explicit inward x2 profile while keeping x1 at incumbent
x2_profile_values = np.linspace(
    lower[1],
    1.0,
    30000
)

x2_profile_candidates = np.column_stack([
    np.full_like(
        x2_profile_values,
        best_x[0]
    ),
    x2_profile_values
])


candidates = np.vstack([
    local_candidates,
    boundary_candidates,
    x2_profile_candidates
])


# ------------------------------------------------------------
# 9. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(
    X
)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "\nCandidates after duplicate filtering:"
)

print(
    len(candidates)
)


# ------------------------------------------------------------
# 10. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 11. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(
    EI
)

print("\n================================")
print("PRIMARY EI")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "mean =",
    mu[ei_idx]
)

print(
    "std =",
    sigma[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)


# ------------------------------------------------------------
# 12. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "mean =",
    mu[mean_idx]
)

print(
    "std =",
    sigma[mean_idx]
)


# ------------------------------------------------------------
# 13. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta
        * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"beta={beta}",
        "\n candidate =",
        candidates[idx],
        "\n mean =",
        round(
            mu[idx],
            6
        ),
        "\n std =",
        round(
            sigma[idx],
            6
        ),
        "\n UCB =",
        round(
            UCB[idx],
            6
        ),
        "\n"
    )


# ------------------------------------------------------------
# 14. Distance from incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta
        * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 15. Boundary status
# ------------------------------------------------------------

def domain_boundary_status(
    x,
    tol=0.005
):

    status = []

    if x[0] <= tol:
        status.append(
            "x1~0"
        )

    elif x[0] >= 1 - tol:
        status.append(
            "x1~1"
        )

    if x[1] <= tol:
        status.append(
            "x2~0"
        )

    elif x[1] >= 1 - tol:
        status.append(
            "x2~1"
        )

    if not status:
        return "interior"

    return ", ".join(
        status
    )


print("\n================================")
print("DOMAIN BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    domain_boundary_status(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    domain_boundary_status(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta
        * sigma
    )

    idx = np.argmax(
        UCB
    )

    print(
        f"UCB beta={beta}:",
        domain_boundary_status(
            candidates[idx]
        )
    )


# ------------------------------------------------------------
# 16. x2 = 1 boundary profile
# ------------------------------------------------------------

boundary_distance, _ = tree.query(
    boundary_candidates,
    k=1
)

boundary_valid = boundary_candidates[
    boundary_distance > 0.01
]

boundary_mu, boundary_sigma = gp.predict(
    boundary_valid,
    return_std=True
)

boundary_idx = np.argmax(
    boundary_mu
)

print("\n================================")
print("x2 = 1 BOUNDARY PROFILE")
print("================================")

print(
    "Best boundary candidate =",
    boundary_valid[boundary_idx]
)

print(
    "mean =",
    boundary_mu[boundary_idx]
)

print(
    "std =",
    boundary_sigma[boundary_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        boundary_valid[boundary_idx]
        - best_x
    )
)


# ------------------------------------------------------------
# 17. x2 inward profile
# ------------------------------------------------------------

profile_distance, _ = tree.query(
    x2_profile_candidates,
    k=1
)

profile_valid = x2_profile_candidates[
    profile_distance > 0.01
]

profile_mu, profile_sigma = gp.predict(
    profile_valid,
    return_std=True
)

profile_idx = np.argmax(
    profile_mu
)

print("\n================================")
print("x2 INWARD PROFILE")
print("================================")

print(
    "Best inward candidate =",
    profile_valid[profile_idx]
)

print(
    "mean =",
    profile_mu[profile_idx]
)

print(
    "std =",
    profile_sigma[profile_idx]
)

print(
    "distance from incumbent =",
    np.linalg.norm(
        profile_valid[profile_idx]
        - best_x
    )
)

DATA
X shape: (21, 2)
Y shape: (21,)

Current best:
[0.711177 1.      ] -> 0.679662733033218

Y range:
min = -0.06562362443733738
max = 0.679662733033218
std = 0.2501938887397335

WEEK 11 CALIBRATION CHECK
Predicted mean: 0.626125645
Predicted std : 0.057152756
Actual        : 0.5895778573315174

Prediction error: -0.036547787668482656

Error / predicted std: -0.6394755078562205

RECENT HIGH-PERFORMING POINTS
Current incumbent: [0.711177 1.      ] -> 0.679662733033218

Week 10: [0.702316 0.995076] -> 0.6682000915555967

Week 11: [0.702314 0.97205 ] -> 0.5895778573315174


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.09**2 * Matern(length_scale=[0.062, 2], nu=2.5) + WhiteKernel(noise_level=0.0405)

ARD lengthscales:
[0.06203222 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.96991695 0.03008305]

EMPIRICAL LOCAL SCALE
Nearest observed point: 0.010009993106890781

Empirical cap: 0.01501498966033617

WEEK 12 TRUST REGION
Centre:
[0.711177 1.      ]

Half-widths:
[0.01240644 0.01501499]

Lower:
[0.69877056 0.98498501]

Upper:
[0.72358344 1.        ]

Candidates after duplicate filtering:
38098

PRIMARY EI
candidate = [0.71995124 0.99507667]
mean = 0.5858213872077758
std = 0.060605757386968884
EI = 0.0015892043667881384

HIGHEST PREDICTED MEAN
candidate = [0.71995124 0.99507667]
mean = 0.5858213872077758
std = 0.060605757386968884

UCB DIAGNOSTICS

beta=0.05 
 candidate = [0.71995124 0.99507667] 
 mean = 0.585821 
 std = 0.060606 
 UCB = 0.588852 

beta=0.1 
 candidate = [0.71995124 0.99507667] 
 mean = 0.585821 
 std = 0.060606 
 UCB = 0.591882 

beta=0.25 
 candi

ValueError: Found array with 0 sample(s) (shape=(0, 2)) while a minimum of 1 is required by GaussianProcessRegressor.

In [2]:
# ============================================================
# FINAL FUNCTION 2 - WEEK 12 SELECTION
# ============================================================
#
# Week 11 showed that moving farther inward in x2 reduced
# performance substantially.
#
# Recent observations provide strong empirical evidence that
# the x2 = 1 boundary remains the best local region.
#
# The updated GP also supports moving x1 upward, and the
# explicit x2=1 boundary profile places the optimum just above
# x1 = 0.72117.
#
# Use the nearest six-decimal point that is safely more than
# 0.01 from the existing incumbent.
# ============================================================

week12_candidate = np.array([
    0.721178,
    1.000000
])

print("================================")
print("FUNCTION 2 - WEEK 12 FINAL")
print("================================")

print("\nCandidate:")
print(week12_candidate)

# GP prediction
final_mu, final_sigma = gp.predict(
    week12_candidate.reshape(1, -1),
    return_std=True
)

print("\nPredicted mean:")
print(final_mu[0])

print("\nPredicted std:")
print(final_sigma[0])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week12_candidate - best_x
    )
)

# Check nearest observation distance
nearest_dist, _ = tree.query(
    week12_candidate.reshape(1, -1),
    k=1
)

print("\nNearest observed-point distance:")
print(nearest_dist[0])

portal = "-".join(
    f"{x:.6f}"
    for x in week12_candidate
)

print("\nPortal format:")
print(portal)

FUNCTION 2 - WEEK 12 FINAL

Candidate:
[0.721178 1.      ]

Predicted mean:
0.5810250877286507

Predicted std:
0.061746696233092944

Distance from current best:
0.010001000000000038

Nearest observed-point distance:
0.010001000000000038

Portal format:
0.721178-1.000000
